In [3]:

# ============================================================
# F-01 BNPL RISK SCORING ENGINE
# STEP 2: GENERATE 30,000 SYNTHETIC APPLICANTS
# ============================================================

import pandas as pd
import numpy as np
import os

# Reproducibility
np.random.seed(42)

# Create data directory
os.makedirs('../data', exist_ok=True)

# Dataset size
N = 30000

print(f"Generating {N:,} BNPL applicants...")

# ------------------------------------------------------------
# 1. APPLICANT INFORMATION
# ------------------------------------------------------------

app_ids = [
    f"APP{i:07d}" for i in range(N)
]

age = np.random.randint(21, 58, N)

monthly_income = np.exp(
    np.random.normal(10.3, 0.55, N)
).clip(12000, 350000)

bureau_score = np.clip(
    np.random.normal(680, 95, N),
    300,
    900
).round(0)

existing_loans = np.random.poisson(1.1, N)

# Debt-to-income percentage
dti_ratio = np.clip(
    np.random.beta(2, 5, N) * 90,
    2,
    95
)

# BNPL tenure
tenure_months = np.random.choice(
    [1, 3, 6, 9, 12],
    N,
    p=[0.35, 0.30, 0.20, 0.10, 0.05]
)

# Purchase amount
order_value = np.exp(
    np.random.normal(7.3, 0.85, N)
).clip(300, 60000)

# Historical repayment punctuality
on_time_pct_hist = np.clip(
    np.random.beta(6, 1.3, N) * 100,
    0,
    100
)

# Device age
device_tenure_days = np.random.exponential(
    420, N
).clip(1, 3000)

# BNPL applications in last 30 days
app_velocity_30d = np.random.poisson(1.6, N)

# City classification
city_tier = np.random.choice(
    ['Tier1', 'Tier2', 'Tier3'],
    N,
    p=[0.40, 0.35, 0.25]
)


# ------------------------------------------------------------
# 2. TRUE DEFAULT PROBABILITY
# ------------------------------------------------------------

logit = (
    -2.1
    - 0.0062 * (bureau_score - 680)
    + 0.021 * dti_ratio
    - 0.000011 * (monthly_income - 80000)
    - 0.019 * on_time_pct_hist
    + 0.14 * existing_loans
    + 0.09 * app_velocity_30d
    - 0.0006 * device_tenure_days
    + np.random.normal(0, 0.6, N)
)

# Convert logit into probability
pd_true = 1 / (1 + np.exp(-logit))

# Generate default labels
default = (
    np.random.random(N) < pd_true
).astype(int)


# ------------------------------------------------------------
# 3. CREATE DATAFRAME
# ------------------------------------------------------------

df = pd.DataFrame({

    'applicant_id': app_ids,

    'age': age,

    'monthly_income': monthly_income.round(0),

    'bureau_score': bureau_score,

    'existing_loans': existing_loans,

    'dti_ratio': dti_ratio.round(1),

    'tenure_months': tenure_months,

    'order_value': order_value.round(0),

    'on_time_pct_hist': on_time_pct_hist.round(1),

    'device_tenure_days': device_tenure_days.round(0),

    'app_velocity_30d': app_velocity_30d,

    'city_tier': city_tier,

    'default': default

})


# ------------------------------------------------------------
# 4. SAVE DATASET
# ------------------------------------------------------------

df.to_csv(
    '../data/bnpl_applicants.csv',
    index=False
)


# ------------------------------------------------------------
# 5. DATASET VALIDATION
# ------------------------------------------------------------

print(f"Shape:              {df.shape}")

print(
    f"Default rate:       {df['default'].mean()*100:.2f}%"
)

print(
    f"Bureau score range: "
    f"{df.bureau_score.min():.0f}-"
    f"{df.bureau_score.max():.0f}"
)

print("Saved: ../data/bnpl_applicants.csv")


Generating 30,000 BNPL applicants...
Shape:              (30000, 13)
Default rate:       10.04%
Bureau score range: 300-900
Saved: ../data/bnpl_applicants.csv


In [4]:
print(df.head())

print("\nDataset Information:")
print(df.info())

print("\nMissing Values:")
print(df.isnull().sum())

print("\nDefault Distribution:")
print(df['default'].value_counts())

print("\nDefault Percentage:")
print(df['default'].value_counts(normalize=True) * 100)

  applicant_id  age  monthly_income  bureau_score  existing_loans  dti_ratio  \
0   APP0000000   49         25796.0         720.0               1       15.7   
1   APP0000001   35         51414.0         717.0               0       30.8   
2   APP0000002   28         41074.0         663.0               1       35.3   
3   APP0000003   41         48058.0         634.0               1       40.3   
4   APP0000004   39         16176.0         763.0               1       46.5   

   tenure_months  order_value  on_time_pct_hist  device_tenure_days  \
0              6        820.0              66.7                10.0   
1              3       3515.0              88.5               198.0   
2              1       2092.0              84.5               466.0   
3              6       1634.0              82.4               834.0   
4              9       1280.0              96.1               396.0   

   app_velocity_30d city_tier  default  
0                 1     Tier2        0  
1         